In [2]:
!pip install kagglehub sentence-transformers


!pip install --no-binary scikit-surprise scikit-surprise

import numpy as np
import pandas as pd
import kagglehub
import torch
import os
from sentence_transformers import SentenceTransformer

print(f"\n✅ Environment Ready!")
print(f"NumPy version: {np.__version__}")
print(f"Torch Version: {torch.__version__}")


✅ Environment Ready!
NumPy version: 2.0.2
Torch Version: 2.10.0+cpu


Dataset Retrieval

In [3]:
import kagglehub
import os

# 1. Download Letterboxd Dataset from Kaggle
print("Downloading Letterboxd dataset (SVD source)...")
kaggle_path = kagglehub.dataset_download("samlearner/letterboxd-movie-ratings-data/versions/3")
print(f"✅ Kaggle data located at: {kaggle_path}")

# 2. Download Tag Genome 2021 Dataset
print("\nDownloading Tag Genome 2021 (1.8GB zip)... this may take a moment.")
tg_url = "https://files.grouplens.org/datasets/tag-genome-2021/genome_2021.zip"
!wget -nc {tg_url}

print("\nUnzipping Tag Genome...")
!unzip -n genome_2021.zip

print("\n--- File Verification ---")
files_to_check = [
    'raw/metadata.json',
    'raw/tags.json',
    'scores/tagdl.csv'
]

for f in files_to_check:
    if os.path.exists(f):
        print(f"✅ Found: {f}")
    else:
        print(f"❌ Missing: {f}")

ratings_export_path = os.path.join(kaggle_path, "ratings_export.csv")
movie_data_path = os.path.join(kaggle_path, "movie_data.csv")

if os.path.exists(ratings_export_path):
    print(f"✅ Found Kaggle Ratings: {ratings_export_path}")

100%|██████████| 108M/108M [00:00<00:00, 157MB/s]

Extracting files...


✅ Kaggle data located at: /root/.cache/kagglehub/datasets/samlearner/letterboxd-movie-ratings-data/versions/3

--2026-04-10 00:55:18--  https://files.grouplens.org/datasets/tag-genome-2021/genome_2021.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1928028583 (1.8G) [application/zip]
Saving to: ‘genome_2021.zip’

genome_2021.zip     100%[===================>]   1.79G  44.6MB/s    in 37s     

2026-04-10 00:55:55 (50.2 MB/s) - ‘genome_2021.zip’ saved [1928028583/1928028583]


Unzipping Tag Genome...
Archive:  genome_2021.zip
   creating: movie_dataset_public_final/
  inflating: movie_dataset_public_final/.DS_Store  
  inflating: __MACOSX/movie_dataset_public_final/._.DS_Store  
   creating: movie_dataset_public_final/scores/
  inflating: movie_dataset_public_final/readme.txt  
  inflating: __MACOSX/movie_dataset_public

Data Alignment and Semantic Similarity

In [8]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer

TG_BASE = "movie_dataset_public_final"
tg_metadata_path = f"{TG_BASE}/raw/metadata.json"
tg_tags_path = f"{TG_BASE}/raw/tags.json"

#Load Kaggle Movie Data (Slugs)
print("Loading Kaggle movie data...")
kaggle_movies = pd.read_csv(movie_data_path)
kaggle_movies['norm_title'] = kaggle_movies['movie_title'].str.lower().str.strip()

# 3. Load Tag Genome Metadata
print("Loading Tag Genome metadata...")
with open(tg_metadata_path, 'r') as f:
    tg_meta = [json.loads(line) for line in f]
tg_df = pd.DataFrame(tg_meta)

# Normalize TG titles (remove years like "(1995)" to match Kaggle format)
tg_df['norm_title'] = tg_df['title'].str.replace(r'\s\(\d{4}\)', '', regex=True).str.lower().str.strip()

# Build the ID Bridge (Slug <-> Item_ID)
print("Building the ID Bridge...")
bridge = pd.merge(
    kaggle_movies[['movie_id', 'norm_title']],
    tg_df[['item_id', 'norm_title']],
    on='norm_title',
    how='inner'
).drop_duplicates('movie_id')

slug_to_itemid = dict(zip(bridge['movie_id'], bridge['item_id']))
itemid_to_slug = dict(zip(bridge['item_id'], bridge['movie_id']))
itemid_to_title = dict(zip(tg_df['item_id'], tg_df['title']))

# Load and Embed Tags
print("\nInitializing SentenceTransformer (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading and embedding tags...")
with open(tg_tags_path, 'r') as f:
    tg_tags = [json.loads(line) for line in f]
tags_df = pd.DataFrame(tg_tags)

tag_texts = tags_df['tag'].tolist()
tag_ids = tags_df['id'].tolist()
tag_id_to_text = dict(zip(tag_ids, tag_texts))

tag_embeddings = embedder.encode(tag_texts, convert_to_tensor=True, show_progress_bar=True)

print(f"\n✅ Bridge built for {len(bridge)} movies.")
print(f"✅ {len(tag_texts)} tags embedded and ready for semantic search.")

Loading Kaggle movie data...
Loading Tag Genome metadata...
Building the ID Bridge...

Initializing SentenceTransformer (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading and embedding tags...


Batches:   0%|          | 0/35 [00:00<?, ?it/s]


✅ Bridge built for 80782 movies.
✅ 1094 tags embedded and ready for semantic search.


Loading Relevance Scores

In [9]:
# Path to the scores file
scores_path = f"{TG_BASE}/scores/tagdl.csv"

print("Loading TagDL relevance scores (this is the heavy part)...")
# We only keep rows for movies that are in our 'bridge' to save RAM
# We use float32 for the scores and int32 for IDs to save memory
tag_rel = pd.read_csv(
    scores_path,
    dtype={'tag': 'category', 'item_id': 'int32', 'score': 'float32'}
)

# Filter the scores to only include our mapped movies
print("Filtering scores to mapped movies...")
initial_count = len(tag_rel)
valid_item_ids = set(slug_to_itemid.values())
tag_rel = tag_rel[tag_rel['item_id'].isin(valid_item_ids)].reset_index(drop=True)

print(f"\n✅ Relevance scores loaded and filtered.")
print(f"Kept {len(tag_rel)} scores out of {initial_count} total.")
print(f"Memory usage: {tag_rel.memory_usage().sum() / 1e6:.2f} MB")

Loading TagDL relevance scores (this is the heavy part)...
Filtering scores to mapped movies...

✅ Relevance scores loaded and filtered.
Kept 6685027 scores out of 10551655 total.
Memory usage: 66.89 MB


Repair

In [ ]:
# 1. Force install the correct NumPy for Python 3.12
!pip install "numpy<2.0.0" --force-reinstall

# 2. Reinstall scikit-surprise so it links to the new library correctly
!pip install --upgrade scikit-surprise

import numpy as np
print(f"NumPy version is now: {np.__version__} (Target: 1.26.4)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requ

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [4]:
import os
import json
import pandas as pd
import kagglehub
from surprise import Dataset, Reader, SVD

# 1. RE-ESTABLISH PATHS (Since the restart cleared them)
print("Locating datasets...")
kaggle_path = kagglehub.dataset_download("samlearner/letterboxd-movie-ratings-data/versions/3")
ratings_export_path = os.path.join(kaggle_path, "ratings_export.csv")
movie_data_path = os.path.join(kaggle_path, "movie_data.csv")
TG_BASE = "movie_dataset_public_final"

# 2. RE-BUILD THE BRIDGE (Needed for the filtering step)
print("Re-building the ID Bridge...")
kaggle_movies = pd.read_csv(movie_data_path)
kaggle_movies['norm_title'] = kaggle_movies['movie_title'].str.lower().str.strip()

with open(f"{TG_BASE}/raw/metadata.json", 'r') as f:
    tg_meta = [json.loads(line) for line in f]
tg_df = pd.DataFrame(tg_meta)
tg_df['norm_title'] = tg_df['title'].str.replace(r'\s\(\d{4}\)', '', regex=True).str.lower().str.strip()

bridge = pd.merge(
    kaggle_movies[['movie_id', 'norm_title']],
    tg_df[['item_id', 'norm_title']],
    on='norm_title',
    how='inner'
).drop_duplicates('movie_id')
valid_slugs = set(bridge['movie_id'].unique())

# 3. CHECK FOR USER HISTORY
USER_RATINGS_FILE = "user_ratings.csv"
RANDOM_SEED = 12

if os.path.exists(USER_RATINGS_FILE):
    print(f"Loading user history from {USER_RATINGS_FILE}...")
    user_ratings = pd.read_csv(USER_RATINGS_FILE)
    target_user_id = user_ratings["user_id"].iloc[0]
    has_user_history = len(user_ratings) >= 5
else:
    print("⚠️ user_ratings.csv not found. Personalization will be inactive.")
    has_user_history = False

# 4. LOAD AND FILTER GLOBAL RATINGS
print("Loading and filtering global ratings (this may take a minute)...")
full_ratings = pd.read_csv(ratings_export_path, usecols=["user_id", "movie_id", "rating_val"])
# Keep only ratings for the 80k movies in our bridge
full_ratings = full_ratings[full_ratings['movie_id'].isin(valid_slugs)]

# 5. COMBINE DATASETS
if has_user_history:
    print(f"Integrating ratings for user '{target_user_id}'...")
    user_data_clean = user_ratings[["user_id", "movie_id", "rating_val"]]
    combined_data = pd.concat([full_ratings, user_data_clean], ignore_index=True)
else:
    combined_data = full_ratings

combined_data.drop_duplicates(inplace=True)

# 6. BUILD AND TRAIN SVD
print(f"Training SVD model on {len(combined_data)} ratings...")
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(combined_data[["user_id", "movie_id", "rating_val"]], reader)
trainset = data.build_full_trainset()

algo = SVD(random_state=RANDOM_SEED)
algo.fit(trainset)

print("\n✅ SVD Personalization Model trained successfully!")

Locating datasets...
Using Colab cache for faster access to the 'letterboxd-movie-ratings-data' dataset.
Re-building the ID Bridge...
Loading user history from user_ratings.csv...
Loading and filtering global ratings (this may take a minute)...
Integrating ratings for user 'crushedoreos'...
Training SVD model on 6231326 ratings...

✅ SVD Personalization Model trained successfully!


Hybrid Recommender

In [11]:
import pandas as pd
import torch
import json
import os
from sentence_transformers import SentenceTransformer, util

# --- 1. RE-ESTABLISH ENVIRONMENT & MAPPINGS ---
print("Synchronizing recommendation engines...")
TG_BASE = "movie_dataset_public_final"

# Re-load Tags and Metadata
with open(f"{TG_BASE}/raw/tags.json", 'r') as f:
    tg_tags = [json.loads(line) for line in f]
tags_df = pd.DataFrame(tg_tags)
tag_texts = tags_df['tag'].tolist()
tag_ids = tags_df['id'].tolist()

with open(f"{TG_BASE}/raw/metadata.json", 'r') as f:
    tg_meta = [json.loads(line) for line in f]
tg_df = pd.DataFrame(tg_meta)
itemid_to_title = dict(zip(tg_df['item_id'], tg_df['title']))

# Re-load Embedder and Generate Tag Vectors
embedder = SentenceTransformer('all-MiniLM-L6-v2')
tag_embeddings = embedder.encode(tag_texts, convert_to_tensor=True)

# Re-load the main Relevance Scores (using 'tag' as the key column)
print("Loading vibe scores...")
tag_rel = pd.read_csv(
    f"{TG_BASE}/scores/tagdl.csv",
    dtype={'tag': 'category', 'item_id': 'int32', 'score': 'float32'}
)

# --- 2. THE HYBRID FUNCTION (Fixed for Schema Mismatch) ---
def get_hybrid_recommendations(user_prompt, n=5):
    # Phase A: Semantic Mood Search
    query_vec = embedder.encode(user_prompt, convert_to_tensor=True)
    cos_scores = util.pytorch_cos_sim(query_vec, tag_embeddings)[0]

    # Get the Top 10 matching tag STRINGS
    top_k_indices = torch.topk(cos_scores, k=10).indices.cpu().numpy()
    matched_tag_names = [tag_texts[idx] for idx in top_k_indices]

    # Phase B: Mood Scoring (Fixed column from 'tag_id' to 'tag')
    # tagdl.csv uses the tag string as the identifier
    mood_matches = tag_rel[tag_rel['tag'].isin(matched_tag_names)].copy()
    movie_mood_scores = mood_matches.groupby('item_id')['score'].sum().reset_index()
    movie_mood_scores.rename(columns={'score': 'mood_score'}, inplace=True)

    # Phase C: Personalization & Combination
    final_results = []
    for _, row in movie_mood_scores.iterrows():
        item_id = int(row['item_id'])
        slug = itemid_to_slug.get(item_id)
        if not slug: continue

        # Calculate predicted preference for 'crushedoreos'
        personal_pred = algo.predict(target_user_id, slug).est if has_user_history else 5.0

        # Combined score: Mood Match * Scaled Personal Preference
        hybrid_score = row['mood_score'] * (personal_pred / 10.0)

        final_results.append({
            'title': itemid_to_title.get(item_id, "Unknown"),
            'predicted_rating': personal_pred,
            'mood_score': row['mood_score'],
            'hybrid_score': hybrid_score
        })

    # Sort and return
    return pd.DataFrame(final_results).sort_values(by='hybrid_score', ascending=False).head(n)

# --- 3. RUN THE SEARCH ---
my_mood = "I want a movie about technology, hackers, or corporate surveillance."
print(f"\nSearching for: \"{my_mood}\"...\n")

results = get_hybrid_recommendations(my_mood)
print(results[['title', 'predicted_rating', 'hybrid_score']].to_string(index=False))

Synchronizing recommendation engines...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading vibe scores...

Searching for: "I want a movie about technology, hackers, or corporate surveillance."...

              title  predicted_rating  hybrid_score
Donnie Darko (2001)          7.899348      4.091509
        Moon (2009)          7.533203      3.708841
   Dark City (1998)          7.013696      3.468743
  District 9 (2009)          7.390304      3.288310
     Gattaca (1997)          7.128513      3.285913


Saving the models

In [12]:
import joblib
from surprise.dump import dump

# 1. Save the SVD Model (The Personalization Engine)
dump("hybrid_svd_model.pkl", algo=algo)

# 2. Save the Vibe Engine components (DataFrames, Tensors, and Mappings)
# These are the static parts that take a long time to load/filter
hybrid_state = {
    'tag_rel': tag_rel,
    'tag_embeddings': tag_embeddings,
    'tag_texts': tag_texts,
    'tag_ids': tag_ids,
    'itemid_to_slug': itemid_to_slug,
    'itemid_to_title': itemid_to_title,
    'target_user_id': target_user_id,
    'has_user_history': has_user_history
}

# We use joblib because it's significantly faster for large DataFrames and Tensors
joblib.dump(hybrid_state, "hybrid_vibe_engine.joblib")

print("✅ Hybrid System Saved!")
print("- hybrid_svd_model.pkl (SVD Logic)")
print("- hybrid_vibe_engine.joblib (Tag Genome Logic)")

✅ Hybrid System Saved!
- hybrid_svd_model.pkl (SVD Logic)
- hybrid_vibe_engine.joblib (Tag Genome Logic)


Test

In [14]:
import os
import joblib
import pandas as pd
import torch
from surprise.dump import load
from sentence_transformers import SentenceTransformer, util

# --- 1. RELOAD COMPONENTS ---
print("Loading Hybrid System...")
_, algo = load("hybrid_svd_model.pkl")
state = joblib.load("hybrid_vibe_engine.joblib")

# Unpack the state
tag_rel = state['tag_rel']
tag_embeddings = state['tag_embeddings']
tag_texts = state['tag_texts']
tag_ids = state['tag_ids']
itemid_to_slug = state['itemid_to_slug']
itemid_to_title = state['itemid_to_title']
target_user_id = state['target_user_id']
has_user_history = state['has_user_history']

# Re-initialize the model (it's small and loads fast from cache)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# --- 2. INFERENCE FUNCTION ---
def get_hybrid_recommendations(user_prompt, n=5):
    query_vec = embedder.encode(user_prompt, convert_to_tensor=True)
    cos_scores = util.pytorch_cos_sim(query_vec, tag_embeddings)[0]
    top_k_indices = torch.topk(cos_scores, k=10).indices.cpu().numpy()
    matched_tag_names = [tag_texts[idx] for idx in top_k_indices]

    mood_matches = tag_rel[tag_rel['tag'].isin(matched_tag_names)].copy()
    movie_mood_scores = mood_matches.groupby('item_id')['score'].sum().reset_index()
    movie_mood_scores.rename(columns={'score': 'mood_score'}, inplace=True)

    final_results = []
    for _, row in movie_mood_scores.iterrows():
        item_id = int(row['item_id'])
        slug = itemid_to_slug.get(item_id)
        if not slug: continue

        personal_pred = algo.predict(target_user_id, slug).est if has_user_history else 5.0
        hybrid_score = row['mood_score'] * (personal_pred / 10.0)

        final_results.append({
            'title': itemid_to_title.get(item_id, "Unknown"),
            'predicted_rating': personal_pred,
            'hybrid_score': hybrid_score
        })

    return pd.DataFrame(final_results).sort_values(by='hybrid_score', ascending=False).head(n)

# --- 3. GET RECOMMENDATIONS ---
prompt = "I want a cyberpunk movie."
results = get_hybrid_recommendations(prompt)
print(results[['title', 'predicted_rating', 'hybrid_score']].to_string(index=False))

Loading Hybrid System...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                            title  predicted_rating  hybrid_score
Terminator 2: Judgment Day (1991)          8.014352      4.124144
        Back to the Future (1985)          8.248296      3.891910
              Total Recall (1990)          7.726913      3.810736
Back to the Future Part II (1989)          7.188112      3.712357
                   RoboCop (1987)          7.683236      3.676507
